# Ámbito de Variables en Python
## `global`, `nonlocal` y Closures

Cuando una función intenta usar una variable, Python la busca siguiendo la regla **LEGB**:

| Nivel | Descripción |
|-------|-------------|
| **L**ocal | Dentro de la función actual |
| **E**nclosing | En la función que la contiene (si hay anidamiento) |
| **G**lobal | En el módulo o script principal |
| **B**uilt-in | Funciones y constantes de Python (`len`, `print`, etc.) |

Las palabras clave `global` y `nonlocal` permiten **modificar** variables de niveles superiores.

---
## 1. El problema: por qué necesitamos `global`

Por defecto, si una función **asigna** a una variable, Python la trata como local aunque exista una global con el mismo nombre.

In [1]:
contador = 0

def incrementar():
    contador += 1  # ← Python asume que 'contador' es local

incrementar()  # UnboundLocalError: cannot access local variable 'contador'

UnboundLocalError: local variable 'contador' referenced before assignment

> **¿Por qué?** Python detecta la asignación `contador += 1` y reserva `contador` como variable local, pero al momento de leer su valor (para sumarle 1), aún no fue definida localmente.

---
## 2. Solución con `global`

`global` le indica a Python que la variable pertenece al módulo, no a la función.

In [2]:
contador = 0

def incrementar():
    global contador   # ahora Python sabe que es la variable del módulo
    contador += 1

incrementar()
incrementar()
incrementar()

print(contador)  # 3

3


### Ejemplo más real — modo debug compartido

In [3]:
debug_mode = False

def activar_debug():
    global debug_mode
    debug_mode = True

def log(mensaje):
    if debug_mode:
        print(f"[DEBUG] {mensaje}")

log("conectando...")    # no imprime nada


In [4]:

activar_debug()


In [5]:

log("conectando...")    # [DEBUG] conectando...
log("datos recibidos")  # [DEBUG] datos recibidos

[DEBUG] conectando...
[DEBUG] datos recibidos


> ⚠️ **Advertencia:** `global` funciona, pero tiene un costo: cualquier parte del código puede modificar esa variable, lo que hace al programa difícil de rastrear y depurar. Úsalo con criterio.

---
## 3. Funciones anidadas y `nonlocal`

Cuando una función está **dentro de otra**, puede leer variables de la función exterior, pero no modificarlas — a menos que use `nonlocal`.

In [6]:
def exterior():
    mensaje = "hola"

    def interior():
        nonlocal mensaje   # hace referencia a la variable de 'exterior'
        mensaje = "chau"

    print("antes:", mensaje)  # hola
    interior()
    print("después:", mensaje) # chau

exterior()

antes: hola
después: chau


In [7]:
menasaje

NameError: name 'menasaje' is not defined

---
## 4. Closures — el uso más poderoso de `nonlocal`

Una **closure** es una función que *recuerda* el estado de su entorno aunque la función exterior ya haya terminado.

En el ejemplo siguiente, `gen_count` devuelve la función `contador` (no su resultado). Cada vez que llamamos a `contador()`, modifica su propia copia de `var`.

In [8]:
def gen_count():
    var = 0
    def contador():
        nonlocal var
        var += 1
        return var
    return contador   # retorna la función, no la ejecuta


In [9]:

# Creamos dos contadores independientes
c1 = gen_count()
c2 = gen_count()


In [10]:
type(c1)  # <class 'function'>

function

In [30]:

print(c1())  # 1
print(c1())  # 2
print(c1())  # 3

print(c2())  # 1  ← instancia propia, no comparte estado con c1
print(c2())  # 2

55
56
57
37
38


In [12]:
var

NameError: name 'var' is not defined

> Cada llamada a `gen_count()` crea una **nueva copia** de `var`. `c1` y `c2` son contadores completamente independientes.

### Ejemplo aplicado — generador de IDs únicos

In [ ]:
def gen_id(prefijo):
    numero = 0
    def siguiente():
        nonlocal numero
        numero += 1
        return f"{prefijo}-{numero:04d}"
    return siguiente

nuevo_usuario = gen_id("USR")
nuevo_producto = gen_id("PROD")


USR-0001
USR-0002
PROD-0001
USR-0003


In [38]:

print(nuevo_usuario())    # USR-0001
print(nuevo_usuario())    # USR-0002
print(nuevo_producto())   # PROD-0001
print(nuevo_usuario())    # USR-0003

USR-0022
USR-0023
PROD-0008
USR-0024


---
## 5. Comparativa final

| | `global` | `nonlocal` |
|---|---|---|
| **Alcance** | Variable del módulo | Variable de la función exterior |
| **Cuándo usarlo** | Estado realmente global | Closures / funciones anidadas |
| **Riesgo** | Alto — cualquiera puede modificarla | Bajo — scope acotado |
| **Alternativa** | Pasar parámetros, usar clases | — |

---
## 6. Ejercicios

**Ejercicio 1:** Escribí una función `gen_pila()` que devuelva dos funciones: `push(valor)` y `pop()`, usando una lista como estado interno con `nonlocal`.

In [ ]:
# Tu solución aquí


**Ejercicio 2:** Usando `global`, creá un sistema de log con nivel de verbosidad: `SILENCIOSO`, `NORMAL`, `VERBOSE`. Implementá funciones `set_nivel(n)` y `log(mensaje, nivel)`.

In [ ]:
# Tu solución aquí
